# Tahap 2A: Transformasi Varians & Uji Stasioneritas Awal
**Bagian:** Maulana

Fokus pengerjaan tahap ini:
1. Cek kestabilan varians (apakah perlu transformasi Log / Box-Cox).
2. Uji stasioneritas awal pada deret asli / level (ADF & KPSS).


In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import boxcox
from statsmodels.tsa.stattools import adfuller, kpss

warnings.filterwarnings("ignore")

# Setup path data fleksibel
ROOT = Path.cwd().resolve()
if (ROOT / "data" / "processed").exists():
    DATA_DIR = ROOT / "data" / "processed"
elif Path(r"C:\Users\Lenovo\Downloads\Analisis Data Time Series\data\processed").exists():
    DATA_DIR = Path(r"C:\Users\Lenovo\Downloads\Analisis Data Time Series\data\processed")
else:
    DATA_DIR = Path(r"C:\Users\Lenovo\.gemini\antigravity\scratch\adt-project\data\processed")

# Load data hasil pra-pemrosesan Kunto
df = pd.read_csv(DATA_DIR / "pm25_surakarta_weekly_model.csv", parse_dates=["week_start"])
series = df.set_index("week_start")["pm25"].asfreq("7D")

# Split train (104 minggu) & test (12 minggu)
train = series.iloc[:-12]
test = series.iloc[-12:]

print(f"Total data : {len(series)} minggu")
print(f"Data train : {len(train)} minggu ({train.index.min().strftime('%Y-%m-%d')} s.d. {train.index.max().strftime('%Y-%m-%d')})")
print(f"Data test  : {len(test)} minggu ({test.index.min().strftime('%Y-%m-%d')} s.d. {test.index.max().strftime('%Y-%m-%d')})")


## 1. Cek Kestabilan Varians
Sebelum differencing, periksa apakah varians data stabil atau membesar seiring naiknya level PM2.5.


In [ ]:
# Rolling mean vs rolling std (window 8 minggu)
roll_mean = train.rolling(8, center=True).mean().dropna()
roll_std = train.rolling(8, center=True).std().dropna()
corr, p_corr = stats.pearsonr(roll_mean, roll_std)

# Ringkasan varians per semester (26 minggu)
sem_data = []
for i in range(4):
    seg = train.iloc[i * 26:(i + 1) * 26]
    sem_data.append({
        "Semester": f"Sem {i+1} ({seg.index.min().strftime('%b %y')} - {seg.index.max().strftime('%b %y')})",
        "Mean": seg.mean(),
        "Std": seg.std(),
        "Varians": seg.var()
    })
df_sem = pd.DataFrame(sem_data)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].scatter(roll_mean, roll_std, color="#1d4ed8", alpha=0.7)
ax[0].set_title(f"Rolling Mean vs Std (r = {corr:.3f}, p = {p_corr:.3f})")
ax[0].set_xlabel("Mean PM2.5")
ax[0].set_ylabel("Std PM2.5")

ax[1].bar([f"Sem {i+1}" for i in range(4)], df_sem["Varians"], color="#0891b2", alpha=0.75)
ax[1].set_title("Varians Tiap Semester")
ax[1].set_ylabel("Varians")
plt.tight_layout()
plt.show()

display(df_sem.round(2))


## 2. Uji Box-Cox & Pertimbangan Transformasi
Cek parameter lambda Box-Cox optimal dan bandingkan dengan deret asli serta transformasi log.


In [ ]:
train_bc, lam = boxcox(train)
train_log = np.log(train)

print(f"Lambda Box-Cox optimal: {lam:.4f}")

fig, ax = plt.subplots(3, 1, figsize=(10, 6.5), sharex=True)
ax[0].plot(train.index, train.values, color="#1d4ed8")
ax[0].set_title("Deret Asli (µg/m³)")

ax[1].plot(train.index, train_log.values, color="#059669")
ax[1].set_title("Transformasi Log")

ax[2].plot(train.index, train_bc, color="#d97706")
ax[2].set_title(f"Transformasi Box-Cox (lambda = {lam:.2f})")
plt.tight_layout()
plt.show()

# Rasio varians max/min antar semester
def var_ratio(arr):
    v = [arr[i * 26:(i + 1) * 26].var() for i in range(4)]
    return max(v) / min(v)

df_tf = pd.DataFrame({
    "Skala": ["Deret Asli", "Log", f"Box-Cox (lambda={lam:.2f})"],
    "Varians Total": [train.var(), train_log.var(), np.var(train_bc)],
    "Rasio Var Max/Min Semester": [var_ratio(train.values), var_ratio(train_log.values), var_ratio(train_bc)]
})
display(df_tf.round(3))


**Keputusan Transformasi:**
Meskipun lambda optimal sekitar 0,14, data **tetap menggunakan skala asli (tidak ditransformasi)**.
* Fluktuasi tinggi terjadi karena faktor musiman alami saat musim kemarau, bukan tren pertumbuhan varians eksponensial.
* Memakai skala asli menjaga interpretasi hasil ramalan tetap langsung dalam satuan µg/m³.


## 3. Uji Stasioneritas Awal Deret Level (ADF & KPSS)
Uji apakah deret level asli sudah stasioner atau butuh differencing:
* **ADF:** $H_0$ = tidak stasioner / ada unit root ($p < 0,05$ -> tolak $H_0$, stasioner).
* **KPSS:** $H_0$ = stasioner ($p < 0,05$ -> tolak $H_0$, tidak stasioner).


In [ ]:
def test_stationarity(ts, label):
    adf = adfuller(ts, autolag="AIC")
    kp = kpss(ts, regression="c", nlags="auto")
    return [
        {
            "Data": label,
            "Uji": "ADF",
            "Statistik": round(adf[0], 3),
            "p-value": round(adf[1], 4),
            "Lag": int(adf[2]),
            "Kesimpulan": "Stasioner" if adf[1] < 0.05 else "Tidak stasioner (unit root)"
        },
        {
            "Data": label,
            "Uji": "KPSS",
            "Statistik": round(kp[0], 3),
            "p-value": round(kp[1], 4),
            "Lag": int(kp[2]),
            "Kesimpulan": "Tidak stasioner" if kp[1] < 0.05 else "Stasioner"
        }
    ]

res = test_stationarity(train, "Data Train (n=104)") + test_stationarity(series, "Data Penuh (n=116)")
df_stat = pd.DataFrame(res)
display(df_stat)


## 4. Kesimpulan untuk Tahap 2B (Aldo)
1. **Transformasi varians:** Tidak dilakukan, pemodelan lanjut menggunakan data asli.
2. **Stasioneritas:** Uji ADF pada data penuh menghasilkan $p = 0,2034 > 0,05$ (tidak stasioner), dan ACF meluruh lambat.
3. **Rekomendasi untuk Aldo:** Lakukan **diferensiasi pertama non-musiman ($d=1$)** pada data asli untuk menstabilkan rata-rata.
